In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from scipy.io import wavfile

In [ ]:
stim_dir = os.path.abspath(r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids\stimuli")
stim_pos_fpath = os.path.join(stim_dir, 'Da_Stimulus_44100Hz_pol1.wav')
stim_neg_fpath = os.path.join(stim_dir, 'Da_Stimulus_44100Hz_pol2.wav')

In [ ]:
sr, stim_pos = wavfile.read(stim_pos_fpath)
_, stim_neg  = wavfile.read(stim_neg_fpath)

In [ ]:
duration = len(stim_pos) / sr * 1000  # in ms
time = np.linspace(0, duration, num=len(stim_pos))
print(duration)

In [ ]:
sns.lineplot(stim_pos)

In [ ]:
import matplotlib.pyplot as plt
SMALL  = 12    # tick labels
MEDIUM = 13    # axis labels, colorbar label  
LARGE  = 14    # subplot titles

plt.rcParams.update({
    'font.size':        SMALL,
    'axes.titlesize':   LARGE,
    'axes.labelsize':   MEDIUM,
    'xtick.labelsize':  SMALL,
    'ytick.labelsize':  SMALL,
})

fig, ax = plt.subplots(figsize=(8, 3))
plt.plot(time, 
         stim_pos, 
         color='#8039C6',
         linewidth=1)
plt.plot(time, stim_neg, color='#8039C6')
plt.xlim(0, duration) # Optional: Set x-axis limits
plt.xlabel('Time (ms)')
#plt.ylabel('Amplitude')
plt.yticks([]) # Optional: Hide y-axis ticks
plt.title('/da/ Stimulus Waveform')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.tick_params(direction="in")
# plt.legend(['Polarity 1', 'Polarity 2'])
#plt.grid(True) # Optional: Add a grid
# plt.show()
fig.savefig('stimwaveform.svg', dpi=600)


In [ ]:
from scipy import signal


In [ ]:
frequencies, times, spectrogram = signal.spectrogram(stim_pos, sr, nperseg=512, noverlap= 512*.99, nfft=1024)


In [ ]:
plt.pcolormesh(times, frequencies, 10 * np.log10(spectrogram), cmap='plasma') # Convert to dB
plt.ylabel('Frequency (Hz)')
plt.xlabel('Time (s)')
plt.title('Spectrogram')
plt.ylim([0, 600])
#plt.colorbar(label='Intensity (dB)')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.io import wavfile
from scipy.signal import spectrogram

# Load audio
# rate, data = wavfile.read('/mnt/user-data/uploads/Da_Stimulus_44100Hz_pol1.wav')
data = stim_pos
rate = sr
# Convert to float, normalize
data = data.astype(np.float32) / np.iinfo(np.int16).max

# Compute spectrogram
f, t, Sxx = spectrogram(data, fs=rate, nperseg=1024, noverlap=512, nfft=2048)

# Convert to dB
Sxx_db = 10 * np.log10(Sxx + 1e-10)

# Duration
duration = len(data) / rate

fig, axes = plt.subplots(2, 1, figsize=(10, 7), facecolor='#0e1117')
fig.patch.set_facecolor('#0e1117')

# --- Waveform ---
ax1 = axes[0]
ax1.set_facecolor('#0e1117')
time_axis = np.linspace(0, duration, len(data))
ax1.plot(time_axis * 1000, data, color='#4fc3f7', linewidth=0.8, alpha=0.9)
ax1.set_ylabel('Amplitude', color='#cfd8dc', fontsize=11)
ax1.set_title('Waveform', color='#eceff1', fontsize=12, pad=6)
ax1.tick_params(colors='#90a4ae')
for spine in ax1.spines.values():
    spine.set_edgecolor('#37474f')
ax1.set_xlim(0, duration * 1000)
ax1.set_xlabel('Time (ms)', color='#cfd8dc', fontsize=10)

# --- Spectrogram ---
ax2 = axes[1]
ax2.set_facecolor('#0e1117')
img = ax2.pcolormesh(t * 1000, f/1000, Sxx_db,
                     shading='gouraud',
                     cmap='inferno',
                     vmin=np.percentile(Sxx_db, 10),
                     vmax=Sxx_db.max())

ax2.set_ylabel('Frequency (kHz)', color='#cfd8dc', fontsize=11)
ax2.set_xlabel('Time (ms)', color='#cfd8dc', fontsize=10)
ax2.set_title('Spectrogram', color='#eceff1', fontsize=12, pad=6)
ax2.set_ylim(0, 2)
ax2.tick_params(colors='#90a4ae')
for spine in ax2.spines.values():
    spine.set_edgecolor('#37474f')

cbar = fig.colorbar(img, ax=ax2, pad=0.02)
cbar.set_label('Power (dB)', color='#cfd8dc', fontsize=10)
cbar.ax.tick_params(colors='#90a4ae')
cbar.outline.set_edgecolor('#37474f')

fig.suptitle('Da_Stimulus_44100Hz_pol1.wav', color='#eceff1', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
# plt.savefig('/mnt/user-data/outputs/spectrogram.png', dpi=150, bbox_inches='tight',
#             facecolor='#0e1117')
# print("Saved.")